# 01 — PCA Denoising
## Machine Learning in SPM Tutorial
*RMS AFM & SPM Meeting 2026*

## Goals

By the end of this notebook you will be able to:

- Understand PCA intuitively and why it is useful for SPM data
- Apply PCA to a synthetic hyperspectral SPM dataset
- Visualise component maps and the corresponding spectra for each principal component
- Compare noisy raw data against PCA-denoised reconstructions and quantify the improvement

## What is PCA?

PCA finds the dominant repeating patterns in your data. Given a large collection of measurements, it identifies the directions in which the data varies the most and orders them from most to least important. The first principal component captures the single largest source of variation; the second captures the next largest (and so on), with the constraint that each new direction is orthogonal to all previous ones.

Imagine a dataset of AFM images collected at many different tip–sample conditions. Most of the variation across images is captured by just a few basic 'templates' — perhaps one template describing overall height contrast and another describing edge sharpness. PCA extracts precisely these templates, along with a map of how strongly each template is expressed at every measurement point. In hyperspectral SPM the same idea applies: each pixel has a full spectrum, and PCA separates the few meaningful spectral shapes from the many random noise fluctuations.

Under the hood, PCA is powered by the Singular Value Decomposition (SVD), which factorises the data matrix into orthogonal directions and their associated weights in a single, numerically stable step.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
import numpy as np
import matplotlib.pyplot as plt
from synthetic.generators import make_hyperspectral_spm
from ml.pca_tools import pca_decompose, pca_reconstruct, component_maps
from viz.plotting import set_style, plot_scree, plot_component_maps, compare_images
set_style()
print("Setup complete.")

: 

## Generate Synthetic Hyperspectral Data

In [ ]:
cube, true_spectra, true_maps, freq = make_hyperspectral_spm(
    n_x=32, n_y=32, n_freq=64, n_components=3, noise_level=0.3, random_state=42)
print(f"Data cube shape: {cube.shape}")
print(f"  \u2192 {cube.shape[0]}\u00d7{cube.shape[1]} pixels, {cube.shape[2]} frequency channels")

## Visualise Raw Data

In [ ]:
rng = np.random.default_rng(0)
pixel_idx = rng.integers(0, 32, size=(5, 2))

# Build normalised RGB image from the three true component maps
rgb = np.stack([
    (true_maps[i] - true_maps[i].min()) / (true_maps[i].ptp() + 1e-12)
    for i in range(3)
], axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

im0 = axes[0].imshow(cube[:, :, 32], cmap='viridis', origin='lower')
axes[0].set_title('One frequency slice (noisy)')
axes[0].set_xlabel('x (pixels)')
axes[0].set_ylabel('y (pixels)')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

colors = ['C0', 'C1', 'C2', 'C3', 'C4']
for k, (py, px) in enumerate(pixel_idx):
    axes[1].plot(freq, cube[py, px, :], color=colors[k], alpha=0.8,
                 label=f'pixel ({px},{py})')
axes[1].set_title('Sample pixel spectra (noisy)')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Amplitude')
axes[1].legend(fontsize=7)

axes[2].imshow(rgb, origin='lower')
axes[2].set_title('True component maps (RGB)')
axes[2].set_xlabel('x (pixels)')
axes[2].set_ylabel('y (pixels)')

fig.tight_layout()
plt.show()

## Apply PCA

In [ ]:
scores, components, evr, pca_model = pca_decompose(cube, n_components=20, return_model=True)
print(f"Top 5 components explain {evr[:5].sum()*100:.1f}% of variance")

In [ ]:
fig = plot_scree(evr, n_show=20)
fig.tight_layout()
plt.show()

## Component Maps and Spectra

The first few components capture real signal; later components are noise. In the scree plot above you should see a sharp 'elbow' after the first three components — these correspond to the three spectral sources used to build the synthetic cube. Components beyond the elbow account for a rapidly diminishing fraction of variance and represent measurement noise rather than meaningful physical variation.

In [ ]:
# Spatial maps for the first 4 principal components
maps4 = component_maps(scores, cube.shape, n_components=4)
fig_maps = plot_component_maps(maps4)
plt.show()

# Spectral loadings for the first 4 principal components
fig_spec, ax_spec = plt.subplots(figsize=(8, 4))
for k in range(4):
    ax_spec.plot(freq, components[k], label=f'PC {k+1}', color=f'C{k}')
ax_spec.set_xlabel('Frequency')
ax_spec.set_ylabel('Loading')
ax_spec.set_title('First 4 PC spectra')
ax_spec.legend()
fig_spec.tight_layout()
plt.show()

## Reconstruct (Denoise)

In [ ]:
n_components_list = [1, 2, 3, 5, 10]
reconstructions = {n: pca_reconstruct(cube, n_components=n) for n in n_components_list}

In [ ]:
slice_idx = 32
n_panels = 1 + len(n_components_list)
fig, axes = plt.subplots(1, n_panels, figsize=(3.5 * n_panels, 3.5))

vmin = cube[:, :, slice_idx].min()
vmax = cube[:, :, slice_idx].max()

axes[0].imshow(cube[:, :, slice_idx], cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Original (noisy)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

for ax, n in zip(axes[1:], n_components_list):
    ax.imshow(reconstructions[n][:, :, slice_idx], cmap='viridis', origin='lower',
              vmin=vmin, vmax=vmax)
    ax.set_title(f'{n} PC{"s" if n > 1 else ""}')
    ax.set_xlabel('x')

fig.suptitle('PCA Denoising \u2014 Frequency Slice 32', fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Approximate noiseless reference using half the available frequency channels as rank
n_ref = cube.shape[2] // 2
reference = pca_reconstruct(cube, n_components=n_ref)

rmse_values = []
for n in n_components_list:
    diff = reconstructions[n] - reference
    rmse_values.append(np.sqrt(np.mean(diff ** 2)))

fig_rmse, ax_rmse = plt.subplots(figsize=(6, 4))
ax_rmse.plot(n_components_list, rmse_values, marker='o', linewidth=2, color='C0')
ax_rmse.set_xlabel('N components retained')
ax_rmse.set_ylabel('RMSE')
ax_rmse.set_title('Reconstruction quality')
ax_rmse.set_xticks(n_components_list)
fig_rmse.tight_layout()
plt.show()

## Takeaways

- **PCA effectively separates signal from noise** in hyperspectral SPM data: retaining just the first few components removes the bulk of random noise while preserving the dominant spectral and spatial features.
- **The scree plot guides component selection**: the 'elbow' marks the transition from meaningful components (steeply declining variance) to noise components (nearly flat tail). For this synthetic dataset, 3 components suffice because the data was constructed from 3 sources.
- **PCA assumes linear mixing**: it works best when each pixel spectrum is a linear combination of a small set of basis spectra. Nonlinear tip–sample interactions, strongly varying background, or overlapping peaks with complex coupling may require more advanced decomposition methods (e.g. NMF, ICA, or autoencoders).
- **Over-truncation discards real signal**: keeping too few components can blur fine spatial features or remove weak-but-real spectral peaks, so always validate reconstructions against physical expectations and, where possible, a reference measurement.